In [5]:
import os 
import requests
import time
import pandas as pd
from dotenv import load_dotenv 

load_dotenv()
api_key = os.getenv("FMP_API_KEY")

In [15]:
import os
import requests
import pandas as pd
from datetime import timedelta
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FMP_API_KEY")

symbol = "AAPL"
target_from = "2024-01-01"
current_to = "2026-05-15"
all_articles = []
seen_urls = set()
limit = 250

print(f"--- Starting Dynamic Fetch for {symbol} ---")

while True:
    print(f"Fetching batch ending at: {current_to}...")
    batch_found = False
    last_date_in_batch = None

    for page in range(101):  # Pages 0–100 (API max is page 100)
        params = {
            "symbols": symbol,
            "from": target_from,
            "to": current_to,
            "page": page,
            "limit": limit,
            "apikey": api_key,
        }

        r = requests.get(
            "https://financialmodelingprep.com/stable/news/stock",
            params=params,
        )

        if r.status_code != 200:
            print(f"  HTTP {r.status_code} on page {page}, stopping inner loop.")
            break

        data = r.json()

        if not data:
            print(f"  Empty response on page {page}, stopping inner loop.")
            break

        # Deduplicate on the fly
        new_articles = [a for a in data if a.get("url") not in seen_urls]
        for a in new_articles:
            seen_urls.add(a["url"])
        all_articles.extend(new_articles)

        batch_found = True
        # Track the oldest date seen in this batch (last item = oldest, since API returns newest-first)
        last_date_in_batch = data[-1]["publishedDate"]

        print(f"  Page {page}: got {len(data)} articles, oldest date: {last_date_in_batch}")

        if len(data) < limit:
            # Fewer than a full page — no more pages to fetch in this window
            break

        if page == 100:
            # Hit the page cap; will slide the window using last_date_in_batch
            print("  Hit page 100 cap, sliding window...")
            break

    if not batch_found or not last_date_in_batch:
        print("No more articles found, exiting.")
        break

    # Stop if the oldest article in this batch is at or before our start date
    if last_date_in_batch <= target_from:
        print(f"Reached target_from ({target_from}), done.")
        break

    # Slide the window: set current_to to 1 second before the oldest article we just saw.
    # Keep the full datetime format so the API can filter precisely.
    dt_obj = pd.to_datetime(last_date_in_batch) - timedelta(seconds=1)
    current_to = dt_obj.strftime("%Y-%m-%d %H:%M:%S")

df = pd.DataFrame(all_articles)
print(f"\nTotal unique articles fetched: {len(df)}")

output_path = f"{symbol}_news.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

--- Starting Dynamic Fetch for AAPL ---
Fetching batch ending at: 2026-05-15...
  Page 0: got 250 articles, oldest date: 2026-05-04 10:59:59
  Page 1: got 250 articles, oldest date: 2026-04-27 05:51:02
  Page 2: got 250 articles, oldest date: 2026-04-17 15:26:32
  Page 3: got 250 articles, oldest date: 2026-04-02 07:15:00
  Page 4: got 249 articles, oldest date: 2026-03-16 13:00:40
Fetching batch ending at: 2026-03-16 13:00:39...
  Page 0: got 250 articles, oldest date: 2026-05-04 10:59:59
  Page 1: got 250 articles, oldest date: 2026-04-27 05:51:02
  Page 2: got 250 articles, oldest date: 2026-04-17 15:26:32
  Page 3: got 250 articles, oldest date: 2026-04-02 07:15:00
  Page 4: got 249 articles, oldest date: 2026-03-16 13:00:40
Fetching batch ending at: 2026-03-16 13:00:39...
  Page 0: got 250 articles, oldest date: 2026-05-04 10:59:59
  Page 1: got 250 articles, oldest date: 2026-04-27 05:51:02
  Page 2: got 250 articles, oldest date: 2026-04-17 15:26:32
  Page 3: got 250 articles, o

KeyboardInterrupt: 

In [30]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FMP_API_KEY")

symbol = "AAPL"
target_from = "2024-01-01"
target_to = "2024-05-15"
all_articles = []
seen_urls = set()
limit = 250

print(f"--- Starting Dynamic Fetch for {symbol} ---")

page = 0
while True:
    print(f"target_from: {target_from} | target_to: {target_to} | page: {page}")
    params = {
        "symbols": symbol,
        "from": target_from,
        "to": target_to,
        "page": page,
        "limit": limit,
        "apikey": api_key,
    }

    r = requests.get(
        "https://financialmodelingprep.com/stable/news/stock",
        params=params,
    )

    if r.status_code != 200:
        print(f"HTTP {r.status_code} on page {page}, stopping.")
        break

    data = r.json()

    if not data:
        print(f"Empty response on page {page}, done.")
        break

    # Deduplicate on the fly
    new_articles = [a for a in data if a.get("url") not in seen_urls]
    for a in new_articles:
        seen_urls.add(a["url"])
    all_articles.extend(new_articles)

    dates = [a["publishedDate"] for a in data]
    oldest_date = min(dates)
    newest_date = max(dates)
    print(f"Page {page}: {len(data)} articles | {oldest_date} → {newest_date} | total so far: {len(all_articles)}")

    # Stop if oldest article has reached or passed our start date
    if oldest_date <= target_from:
        print(f"Reached target_from ({target_from}), done.")
        break

    # Partial page = no more results in this window, done
    if len(data) < limit:
        print("Partial page received, done.")
        target_to = oldest_date
        page = 0
        continue

    # Hit page 100 cap — slide the window down and reset pagination
    if page == 100:
        print(f"Hit page 100 cap, sliding window to {oldest_date} and resetting to page 0...")
        target_to = oldest_date
        page = 0
        continue

    page += 1

df = pd.DataFrame(all_articles)
print(f"\nTotal unique articles fetched: {len(df)}")

output_path = f"{symbol}_news.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

--- Starting Dynamic Fetch for AAPL ---
target_from: 2024-01-01 | target_to: 2024-05-15 | page: 0
Page 0: 250 articles | 2024-05-03 06:02:51 → 2024-05-15 20:11:05 | total so far: 250
target_from: 2024-01-01 | target_to: 2024-05-15 | page: 1
Page 1: 249 articles | 2024-04-19 04:00:00 → 2024-05-03 06:02:51 | total so far: 498
Partial page received, done.
target_from: 2024-01-01 | target_to: 2024-04-19 04:00:00 | page: 0
Page 0: 250 articles | 2026-05-04 10:59:59 → 2026-05-15 14:16:50 | total so far: 748
target_from: 2024-01-01 | target_to: 2024-04-19 04:00:00 | page: 1
Page 1: 250 articles | 2026-04-27 05:51:02 → 2026-05-04 10:40:23 | total so far: 998
target_from: 2024-01-01 | target_to: 2024-04-19 04:00:00 | page: 2
Page 2: 250 articles | 2026-04-17 15:26:32 → 2026-04-27 05:08:06 | total so far: 1248
target_from: 2024-01-01 | target_to: 2024-04-19 04:00:00 | page: 3
Page 3: 250 articles | 2026-04-02 07:15:00 → 2026-04-17 13:45:03 | total so far: 1498
target_from: 2024-01-01 | target_to

KeyboardInterrupt: 

In [49]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FMP_API_KEY")

symbol = "AAPL"
target_from = "1990-01-01"
target_to = "2024-05-15" # Must be YYYY-MM-DD
all_articles = []
seen_urls = set()
limit = 250

print(f"--- Starting Dynamic Fetch for {symbol} ---")

page = 0
last_slid_date = None

while True:
    # Ensure date parameters are strictly YYYY-MM-DD
    # We strip any potential time component just in case
    clean_from = target_from.split(' ')[0]
    clean_to = target_to.split(' ')[0]

    params = {
        "symbols": symbol,
        "from": clean_from,
        "to": clean_to,
        "page": page,
        "limit": limit,
        "apikey": api_key,
    }

    try:
        # Using the stable search endpoint
        r = requests.get("https://financialmodelingprep.com/stable/news/stock", params=params)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

    if not data:
        print(f"No more data found for range {clean_from} to {clean_to}. Stopping.")
        break

    # 1. Process and deduplicate
    new_entries_this_page = 0
    for a in data:
        if a.get("url") not in seen_urls:
            seen_urls.add(a["url"])
            all_articles.append(a)
            new_entries_this_page += 1
    
    # 2. Track bounds
    dates = [a["publishedDate"] for a in data]
    oldest_on_page = min(dates)
    newest_on_page = max(dates)
    
    # Logic Check: If the API returns stuff newer than our 'to' date, it's ignoring the filter.
    if newest_on_page.split(' ')[0] > clean_to:
        print(f"CRITICAL: API ignored date filter (Got {newest_on_page} for 'to' date {clean_to}).")
        break

    print(f"Range: {clean_to} | Page {page} | Oldest: {oldest_on_page} | Total: {len(all_articles)}")

    # 3. Termination: Reached the absolute start
    if oldest_on_page.split(' ')[0] <= clean_from:
        print(f"Reached target_from: {clean_from}")
        break

    # 4. Handle Window Sliding
    # If we hit the 100-page limit (0-99) OR if we get no new results (end of window)
    if page >= 99:
        # Move target_to to the DATE of the oldest article found
        new_to_date = oldest_on_page.split(' ')[0]
        
        # Safety: If we are stuck on the same day (more than 25k articles in one day)
        # we subtract one day to force the window to move.
        if new_to_date == last_slid_date:
            dt_obj = datetime.strptime(new_to_date, '%Y-%m-%d')
            new_to_date = (dt_obj - timedelta(days=1)).strftime('%Y-%m-%d')
        
        print(f"--- Sliding window to {new_to_date} and resetting page to 0 ---")
        target_to = new_to_date
        last_slid_date = new_to_date
        page = 0
    else:
        page += 1

# Final Processing
if all_articles:
    df = pd.DataFrame(all_articles)
    df['publishedDate'] = pd.to_datetime(df['publishedDate'])
    df = df.sort_values("publishedDate", ascending=False)
    output_path = f"{symbol}_news_debug.csv"
    df.to_csv(output_path, index=False)
    print(f"\nSaved {len(df)} unique articles to {output_path}")

--- Starting Dynamic Fetch for AAPL ---
Range: 2024-05-15 | Page 0 | Oldest: 2024-05-03 06:02:51 | Total: 250
Range: 2024-05-15 | Page 1 | Oldest: 2024-04-19 04:00:00 | Total: 498
Range: 2024-05-15 | Page 2 | Oldest: 2024-04-01 19:21:01 | Total: 747
Range: 2024-05-15 | Page 3 | Oldest: 2024-03-19 12:00:37 | Total: 997
Range: 2024-05-15 | Page 4 | Oldest: 2024-03-04 10:15:43 | Total: 1247
Range: 2024-05-15 | Page 5 | Oldest: 2024-02-13 09:04:39 | Total: 1495
Range: 2024-05-15 | Page 6 | Oldest: 2024-01-25 16:02:37 | Total: 1743
Range: 2024-05-15 | Page 7 | Oldest: 2024-01-12 10:17:12 | Total: 1992
Range: 2024-05-15 | Page 8 | Oldest: 2023-12-27 06:00:00 | Total: 2242
Range: 2024-05-15 | Page 9 | Oldest: 2023-12-06 09:41:19 | Total: 2492
Range: 2024-05-15 | Page 10 | Oldest: 2023-11-09 14:23:41 | Total: 2742
Range: 2024-05-15 | Page 11 | Oldest: 2023-10-24 15:40:00 | Total: 2989
Range: 2024-05-15 | Page 12 | Oldest: 2023-09-28 19:54:39 | Total: 3238
Range: 2024-05-15 | Page 13 | Oldest: 

In [37]:
df['publishedDate'].min()

'2024-05-03 06:02:51'

In [20]:
data = r.json()

In [21]:
data

[{'symbol': 'AAPL',
  'publishedDate': '2024-03-19 09:37:15',
  'publisher': 'Business Insider',
  'title': "Google and Apple's rumored partnership could upend the AI industry",
  'image': 'https://images.financialmodelingprep.com/news/google-and-apples-rumored-partnership-could-upend-the-ai-20240319.jpg',
  'site': 'businessinsider.com',
  'text': "This post originally appeared in the Insider Today newsletter. You can sign up for Business Insider's daily newsletter here.",
  'url': 'https://www.businessinsider.com/google-apple-partnership-rumors-ai-industry-gemini-model-iphone-2024-3'},
 {'symbol': 'AAPL',
  'publishedDate': '2024-03-19 08:14:00',
  'publisher': 'InvestorPlace',
  'title': '3 Augmented Reality Stocks That Could Soar as AR Goes Mainstream',
  'image': 'https://images.financialmodelingprep.com/news/3-augmented-reality-stocks-that-could-soar-as-ar-20240319.jpg',
  'site': 'investorplace.com',
  'text': "Augmented reality is a powerful technology that doesn't get nearly e